In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
from typing import Tuple, Optional, Dict, Any


# ============================================================================
# CONFIGURATION
# ============================================================================

CHARACTERS = [
    'Intervention', 'Barrier', 'CrossingSignal',
    'Man', 'Woman', 'Pregnant', 'Stroller', 'OldMan', 'OldWoman',
    'Boy', 'Girl', 'Homeless', 'LargeWoman', 'LargeMan', 'Criminal',
    'MaleExecutive', 'FemaleExecutive', 'FemaleAthlete', 'MaleAthlete',
    'FemaleDoctor', 'MaleDoctor', 'Dog', 'Cat'
]

NUM_CHARACTERS = len(CHARACTERS)
MAX_CARDINALITY = 10
NUM_TEAMS = 2


# ============================================================================
# DATASET
# ============================================================================

class MoralChoiceDataset(Dataset):
    def __init__(self, scenarios, labels):
        """
        Args:
            scenarios: (N, 2, 23) array
            labels: (N, 1) array
        """
        # Convert to numpy if needed for memory efficiency
        if isinstance(scenarios, torch.Tensor):
            scenarios = scenarios.numpy()
        if isinstance(labels, torch.Tensor):
            labels = labels.numpy()

        self.scenarios = scenarios
        self.labels = labels

    def __len__(self):
        return len(self.scenarios)

    def __getitem__(self, idx):
        # Convert to tensor only when accessed
        scenario = torch.tensor(self.scenarios[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return scenario, label


# ============================================================================
# MODEL
# ============================================================================

class MoralReasoningTransformer(nn.Module):
    def __init__(
        self,
        num_characters: int = NUM_CHARACTERS,
        max_cardinality: int = MAX_CARDINALITY,
        num_teams: int = NUM_TEAMS,
        embed_dim: int = 128,
        num_heads: int = 8,
        num_layers: int = 6,
        dropout: float = 0.1
    ):
        super().__init__()

        self.num_characters = num_characters
        self.embed_dim = embed_dim

        # Compositional embeddings
        self.character_embedding = nn.Embedding(num_characters, embed_dim)
        self.cardinality_embedding = nn.Embedding(max_cardinality + 1, embed_dim//2)
        self.team_embedding = nn.Embedding(num_teams, embed_dim//2)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # CLS token for aggregation
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # Classification head on CLS token
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, 1)
        )

    def encode_outcome(self, counts, team_id):
        batch_size = counts.shape[0]

        character_ids = torch.arange(
            self.num_characters,
            device=counts.device
        ).unsqueeze(0).expand(batch_size, -1)

        char_emb = self.character_embedding(character_ids)
        card_emb = self.cardinality_embedding(counts)

        team_id_tensor = torch.full(
            (batch_size, self.num_characters),
            team_id,
            device=counts.device,
            dtype=torch.long
        )
        team_emb = self.team_embedding(team_id_tensor)

        tokens = char_emb + card_emb + team_emb

        return tokens

    def forward(self, scenarios):
        batch_size = scenarios.shape[0]

        outcome_0 = scenarios[:, 0, :]
        outcome_1 = scenarios[:, 1, :]

        tokens_0 = self.encode_outcome(outcome_0, team_id=0)
        tokens_1 = self.encode_outcome(outcome_1, team_id=1)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        all_tokens = torch.cat([cls_tokens, tokens_0, tokens_1], dim=1)

        encoded = self.transformer(all_tokens)
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)

        return logits


# ============================================================================
# TRAINING
# ============================================================================

def train_epoch(model, dataloader, optimizer, scheduler, device, epoch, grad_clip=1.0):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]")

    for batch_idx, (scenarios, labels) in enumerate(pbar):
        scenarios = scenarios.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(scenarios)
        loss = F.binary_cross_entropy_with_logits(
            logits.squeeze(-1),
            labels.squeeze(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(logits.squeeze(-1)) > 0.5).float()
        correct += (preds == labels.squeeze(-1)).sum().item()
        total += len(labels)

        # Update progress bar every 10 batches
        if batch_idx % 10 == 0:
            current_acc = correct / total if total > 0 else 0
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{current_acc:.4f}'
            })

    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, device, epoch):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]")

    with torch.no_grad():
        for scenarios, labels in pbar:
            scenarios = scenarios.to(device)
            labels = labels.to(device)

            logits = model(scenarios)
            loss = F.binary_cross_entropy_with_logits(
                logits.squeeze(-1),
                labels.squeeze(-1)
            )

            total_loss += loss.item()
            preds = (torch.sigmoid(logits.squeeze(-1)) > 0.5).float()
            correct += (preds == labels.squeeze(-1)).sum().item()
            total += len(labels)

    return total_loss / len(dataloader), correct / total


# ============================================================================
# MAIN
# ============================================================================

def train_model(
    train_data: Tuple[np.ndarray, np.ndarray],
    val_data: Tuple[np.ndarray, np.ndarray],
    batch_size: int = 32,
    learning_rate: float = 1e-4,
    num_epochs: int = 50,
    weight_decay: float = 0.01,
    embed_dim: int = 128,
    num_heads: int = 8,
    num_layers: int = 6,
    dropout: float = 0.1,
    grad_clip: float = 1.0,
    scheduler_type: Optional[str] = 'cosine',  # 'cosine', 'step', or None
    warmup_ratio: float = 0.1,
    patience: int = 10,
    save_path: str = 'best_model.pt',
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Train the moral reasoning transformer.

    Args:
        train_data: Tuple of (scenarios, labels) for training
        val_data: Tuple of (scenarios, labels) for validation
        batch_size: Batch size for training
        learning_rate: Peak learning rate
        num_epochs: Maximum number of epochs
        weight_decay: L2 regularization strength
        embed_dim: Embedding dimension
        num_heads: Number of attention heads
        num_layers: Number of transformer layers
        dropout: Dropout probability
        grad_clip: Gradient clipping threshold
        scheduler_type: Learning rate scheduler ('cosine', 'step', or None)
        warmup_ratio: Fraction of training for warmup (if using scheduler)
        patience: Early stopping patience
        save_path: Path to save best model
        verbose: Whether to print detailed logs

    Returns:
        Dictionary containing training history and best metrics
    """
    if verbose:
        print("="*80)
        print("MORAL REASONING TRANSFORMER - TRAINING")
        print("="*80)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if verbose:
        print(f"\n[DEVICE] Training on {device}")
        if torch.cuda.is_available():
            print(f"[GPU] {torch.cuda.get_device_name(0)}")

    # Unpack data
    train_scenarios, train_labels = train_data
    val_scenarios, val_labels = val_data

    if verbose:
        print(f"\n[DATA] Train samples: {len(train_scenarios):,}")
        print(f"[DATA] Val samples: {len(val_scenarios):,}")

    # Create datasets
    train_dataset = MoralChoiceDataset(train_scenarios, train_labels)
    val_dataset = MoralChoiceDataset(val_scenarios, val_labels)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=True,
    )

    if verbose:
        print(f"[DATALOADER] Train batches: {len(train_loader):,}")
        print(f"[DATALOADER] Val batches: {len(val_loader):,}")

    # Initialize model
    model = MoralReasoningTransformer(
        embed_dim=embed_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        dropout=dropout
    ).to(device)

    if verbose:
        num_params = sum(p.numel() for p in model.parameters())
        print(f"\n[MODEL] Total parameters: {num_params:,}")
        print(f"[MODEL] Embedding dim: {embed_dim}")
        print(f"[MODEL] Num heads: {num_heads}")
        print(f"[MODEL] Num layers: {num_layers}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
        betas=(0.9, 0.999)
    )

    # Setup scheduler
    scheduler = None
    if scheduler_type is not None:
        total_steps = len(train_loader) * num_epochs
        warmup_steps = int(total_steps * warmup_ratio)

        if scheduler_type == 'cosine':
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=learning_rate,
                total_steps=total_steps,
                pct_start=warmup_steps / total_steps,
                anneal_strategy='cos'
            )
            if verbose:
                print(f"[SCHEDULER] OneCycleLR (warmup_ratio={warmup_ratio})")
        elif scheduler_type == 'step':
            # Step decay after warmup
            scheduler = torch.optim.lr_scheduler.StepLR(
                optimizer,
                step_size=len(train_loader) * 5,  # Decay every 5 epochs
                gamma=0.5
            )
            if verbose:
                print(f"[SCHEDULER] StepLR (step_size=5 epochs, gamma=0.5)")
    else:
        if verbose:
            print(f"[SCHEDULER] None (constant LR)")

    if verbose:
        print(f"[OPTIMIZER] AdamW (lr={learning_rate}, weight_decay={weight_decay})")
        print("\n" + "="*80)
        print("STARTING TRAINING")
        print("="*80 + "\n")

    best_val_acc = 0
    best_val_loss = float('inf')
    patience_counter = 0

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'learning_rate': []
    }

    for epoch in range(1, num_epochs + 1):
        if verbose:
            print(f"\n{'='*80}")
            print(f"EPOCH {epoch}/{num_epochs}")
            print(f"{'='*80}")

        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, device, epoch, grad_clip
        )
        val_loss, val_acc = evaluate(model, val_loader, device, epoch)

        current_lr = optimizer.param_groups[0]['lr']

        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['learning_rate'].append(current_lr)

        if verbose:
            print(f"\n[RESULTS] Epoch {epoch:03d}/{num_epochs}")
            print(f"  Learning Rate: {current_lr:.2e}")
            print(f"  Train: Loss={train_loss:.4f} | Acc={train_acc:.4f}")
            print(f"  Val:   Loss={val_loss:.4f}   | Acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            patience_counter = 0
            if verbose:
                print(f"  ✓ NEW BEST! Saving model to {save_path}")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'train_loss': train_loss,
                'hyperparameters': {
                    'embed_dim': embed_dim,
                    'num_heads': num_heads,
                    'num_layers': num_layers,
                    'dropout': dropout,
                    'learning_rate': learning_rate,
                    'batch_size': batch_size,
                    'weight_decay': weight_decay
                }
            }, save_path)
        else:
            patience_counter += 1
            if verbose:
                print(f"  No improvement ({patience_counter}/{patience})")
            if patience_counter >= patience:
                if verbose:
                    print(f"\n[EARLY STOP] No improvement for {patience} epochs")
                break

    if verbose:
        print("\n" + "="*80)
        print("TRAINING COMPLETE")
        print(f"Best validation accuracy: {best_val_acc:.4f}")
        print(f"Best validation loss: {best_val_loss:.4f}")
        print("="*80 + "\n")

    return {
        'best_val_acc': best_val_acc,
        'best_val_loss': best_val_loss,
        'final_train_acc': history['train_acc'][-1],
        'final_train_loss': history['train_loss'][-1],
        'history': history,
        'epochs_trained': epoch
    }


In [ ]:
import pickle
import gzip

# Load the compressed file
with gzip.open("moral_machine_vector_data_2.pkl.gz", "rb") as f:
    train_scen, train_lab, val_scen, val_lab = pickle.load(f)

print(f"Training set: {train_scen.shape}")
print(f"Training labels: {train_lab.shape}")
print(f"Validation set: {val_scen.shape}")
print(f"Validation labels: {val_lab.shape}")

In [ ]:
results = train_model(
    train_data=(train_scen, train_lab),
    val_data=(val_scen, val_lab),
    batch_size=512,
    learning_rate=1e-4,
    num_epochs=10)